# Greenfilling trade-offs

An exploratory look at the full 28-day campaign. The goal is to get a feel for what greenfilling (optimizing either carbon or water) does to the environmental footprint and to job performance, compared to plain EASY backfilling on the same workload and the same environmental window.

The campaign samples 36 windows of about four weeks each, spread across three grids (DE, FR, PL) and the four seasons, and runs four workload scenarios on every window: a `stress` and a `slack` extract from each of two machines, Mustang and Trinity. These labels describe how the extracts were selected within each source trace. They are not assumed to be equivalent across machines. The analysis below keeps all four scenarios separate so the comparison stays paired and avoids generalizing from one extract.

This analysis looks at replay jobs separately before saying anything about performance.

## Load Campaign Metadata

In [ ]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

import campaign_analysis as ca

pd.set_option("display.max_columns", 120)
plt.rcParams.update({"figure.dpi": 120, "axes.grid": True, "grid.alpha": 0.25})

root = Path.cwd()
if not (root / "experiments").exists():
    root = root.parent

campaign_path = root / "experiments" / "experiments.toml"
windows_path = root / "intensities" / "windows.csv"
out_dir = root / "experiments" / "out"

print(f"campaign: {campaign_path}")
print(f"windows: {windows_path}")
print(f"outputs: {out_dir}")

In [ ]:
campaign_df, windows = ca.load_campaign(campaign_path, windows_path)

display(campaign_df.head())
display(windows.head())

## Check Result Completeness

In [ ]:
result_files = ca.check_completeness(campaign_df, out_dir)

display(ca.completeness_summary(result_files))

missing = ca.missing_results(result_files)
if missing.empty:
    print(f"All {len(result_files)} experiments have schedule and job outputs.")
else:
    display(missing.sort_values(["variant", "workload_label", "zone", "start_date"]))

## Load Result Tables

This section builds the reusable tables for the rest of the notebook. `schedules` has one row per experiment. `jobs` has one row per simulated job and labels each job as either `context` or `replay`.

For slowdown, the main metric is average bounded slowdown with a 10-second floor: `max(turnaround_time / max(execution_time, 10), 1)`. This keeps very short jobs from dominating the comparison while preserving the direction of the performance trade-off.

In [ ]:
schedules, jobs = ca.load_results(campaign_df, result_files, out_dir, windows)

print(f"schedules: {len(schedules)} rows")
print(f"jobs: {len(jobs)} rows")

display(schedules[["name", "variant", "zone", "start_date", "season", "nb_jobs"]].head())
print(f"bounded slowdown floor: {ca.BOUNDED_SLOWDOWN_FLOOR_SECONDS:g} seconds")
display(jobs[["name", "job_id", "job_kind", "waiting_time", "stretch", "bounded_slowdown"]].head())

In [ ]:
job_metrics = ca.build_job_metrics(jobs)

display(ca.summarize_job_metrics(job_metrics))
display(job_metrics.head())

## Paired comparison: greenfilling versus EASY

The traces differ by country and window, and there are four workloads (two machines, each with a stress and a slack regime), so I compare paired by `(workload, zone, start_date)`: each greenfilling run sits next to the EASY run from the same workload, platform, and window. Most deltas below are percentages relative to EASY in that same workload and window. Median waiting time is an absolute difference in seconds because a zero-second EASY median would make a relative change undefined. Results are kept split by workload throughout so the four workloads are never averaged together.

For performance I look at replay jobs only. Context jobs are the initial backlog, and folding them in would mix warm-up with the behavior I actually care about.

In [ ]:
run_metrics = ca.build_run_metrics(schedules, job_metrics)

display(run_metrics.head())

The table below summarizes paired deltas across the 36 windows. Percentages are relative to EASY in the same window, except median waiting time, which is an absolute difference in seconds.

In [ ]:
paired_comparisons = ca.build_paired_comparisons(run_metrics)

display(ca.summarize_paired(paired_comparisons))
display(
    paired_comparisons[
        ["greenfilling_variant", "workload_label", "zone", "season", "start_date"]
        + ca.DELTA_COLUMNS[1:]
    ].head()
)

## Distributions of deltas versus EASY

Each point is one sampled window, and the box just summarizes the spread across windows. Within each objective the four workloads sit side by side (color encodes workload). The line at zero is "no change from EASY." Below zero means lower footprint, waiting time, or slowdown than EASY.

In [ ]:
ca.plot_environmental_deltas(paired_comparisons);

This is the basic question: does greenfilling move the footprint it targets, and is the direction consistent across the sampled windows? Because the comparison is paired by window, a change in a high-carbon window and one in a low-carbon window each count within their own baseline. The four workload scenarios should be interpreted separately.

In [ ]:
ca.plot_performance_deltas(paired_comparisons);

Here the point is what that footprint change costs on the replay jobs. An environmental gain matters less if it comes with a large increase in waiting time or slowdown, so it is worth checking how the performance cost compares to the footprint movement.

## Environmental saving versus performance penalty

The boxes above show each axis on its own. The next two scatters try to show the trade-off directly: one point per window, environmental saving on x, replay slowdown penalty on y. Rows are the four workloads and columns are zones. Roughly, lower-right is the nice region (less footprint, no extra slowdown), upper-right is a real trade-off, and left of zero means the footprint got worse in that window.

In [ ]:
ca.plot_tradeoff_scatter(
    paired_comparisons,
    "total_carbon_footprint_delta_pct",
    "Carbon footprint saving [% vs EASY]",
    "Carbon saving versus bounded slowdown penalty",
);

Carbon saving on the x-axis. I keep the water-objective points in too, since a scheduler tuned for one signal might still move the other.

In [ ]:
ca.plot_tradeoff_scatter(
    paired_comparisons,
    "total_water_footprint_delta_pct",
    "Water footprint saving [% vs EASY]",
    "Water saving versus bounded slowdown penalty",
);

Same view for water. Putting it beside the carbon figure gives a rough sense of whether optimizing one signal helps or fights the other.

## Energy and intensity exposure

A footprint change can come from consuming a different amount of energy, encountering a different intensity while consuming it, or both. The next figure separates these mechanisms. Consumed energy covers the whole simulated platform. Effective carbon and water intensities are the operational carbon and off-site water footprints divided by that energy. They are energy-weighted averages over the schedule, not simple averages over time.

In [ ]:
ca.plot_energy_exposure_deltas(paired_comparisons);

Reading energy beside effective intensity helps avoid attributing every footprint difference to temporal shifting. A longer schedule can increase platform energy even if it encounters a slightly cleaner or less water-intensive mix. Conversely, an intensity improvement can be offset by extra energy use.

## Does swing line up with the savings?

In [ ]:
display(ca.summarize_swing_correlations(paired_comparisons))
ca.plot_swing_relationship(paired_comparisons);

`swing` is the mean daily peak-to-trough range of a signal over the window mean, so it is a rough proxy for how much daily variation the signal has. The working hypothesis is that more variation may give the scheduler more opportunity to change intensity exposure. Each panel pairs a signal's swing with greenfilling's saving for the matching objective. All four workloads share the panel, marker shape encodes workload, and the dashed least-squares fit and `r` pool the four scenarios. The pooled points reuse each environmental window four times, so the correlation is descriptive rather than four independent observations per window. Workload-specific relationships should also be inspected.

## Takeaway

Interpret the results as four selected workload scenarios rather than estimates for Mustang, Trinity, or HPC systems in general. For each scenario, first check whether the target footprint moves consistently across windows. Then compare total footprint with consumed energy and effective intensity to distinguish extra consumption from temporal exposure. Finally, read that environmental change beside waiting time, bounded slowdown, and makespan.

Differences between the carbon and water objectives can indicate whether the two signals offer different scheduling opportunities. Differences between stress and slack can indicate sensitivity to the selected workload regime. Neither comparison by itself identifies a general causal effect because there is one extract per machine-regime combination.